# Financial Data Validation

Run this notebook after loading the regenerated CSV files into PostgreSQL using the updated `sql/02_schema_setup.sql`. The `orders` table now requires the unique `order_line_id` column. The validation uses synthetic costs, invoices, payments, and inventory as documented in `docs/metric_dictionary.md`.


In [ ]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path
from getpass import getpass

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sqlalchemy.engine import *



print("Libraries imported successfully.")

In [ ]:
# Resolve paths from the project root or scripts/ directory (portable across machines).
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "scripts":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
DOCS_DIR = PROJECT_DIR / "docs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directories configured.")
print("Data directory:", DATA_DIR)
print("Documentation directory:", DOCS_DIR)


In [ ]:
DB_USER = input("Enter PostgreSQL username: ")
DB_PASSWORD = getpass("Enter PostgreSQL password: ")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host="localhost",
    port=5432,
    database="retail_analytics"
)

engine = create_engine(connection_url)

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT version();"))
        print("PostgreSQL connected successfully!")
        print(result.fetchone()[0])

except Exception as e:
    print("Database connection failed:", e)
    raise


In [ ]:
products_df = pd.read_sql_query(
    "SELECT * FROM products",
    engine
)

orders_df = pd.read_sql_query(
    "SELECT * FROM orders",
    engine
)

invoices_df = pd.read_sql_query(
    "SELECT * FROM invoices",
    engine
)

payments_df = pd.read_sql_query(
    "SELECT * FROM payments",
    engine
)

print("All tables loaded successfully!")

In [ ]:
datasets = {
    "Products": products_df,
    "Orders": orders_df,
    "Invoices": invoices_df,
    "Payments": payments_df
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("Missing values:", df.isnull().sum().sum())

In [ ]:
required_columns = {
    "products": ["product_id", "product_name", "unit_cost"],
    "orders": [
        "order_line_id",
        "order_id",
        "product_id",
        "unit_selling_price"
    ],
    "invoices": [
        "invoice_id",
        "invoice_amount"
    ],
    "payments": [
        "payment_id",
        "invoice_id",
        "payment_amount"
    ]
}

for table_name, columns in required_columns.items():

    df = datasets[table_name.title()]

    missing_columns = [
        col for col in columns if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing columns: {missing_columns}"
        )

# Standardize identifiers as strings
for df, columns in [
    (products_df, ["product_id"]),
    (orders_df, ["order_id", "product_id"]),
    (invoices_df, ["invoice_id"]),
    (payments_df, ["payment_id", "invoice_id"])
]:
    for col in columns:
        df[col] = df[col].astype("string").str.strip()

# Standardize numerical columns
for df, columns in [
    (products_df, ["unit_cost"]),
    (orders_df, ["unit_selling_price"]),
    (invoices_df, ["invoice_amount"]),
    (payments_df, ["payment_amount"])
]:
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

assert orders_df["order_line_id"].is_unique, "order_line_id must be unique"

print("Column and datatype validation completed.")

In [ ]:
EXCEPTION_COLUMNS = [
    "record_id",
    "rule_failed",
    "observed_value",
    "reason",
    "resolution_status"
]

def create_exception_df(
    record_id,
    rule_failed,
    observed_value,
    reason
):
    exception_df = pd.DataFrame({
        "record_id": record_id,
        "rule_failed": rule_failed,
        "observed_value": observed_value,
        "reason": reason,
        "resolution_status": "Needs Review"
    })

    return exception_df[EXCEPTION_COLUMNS]

print("Exception logging structure initialized.")

In [ ]:
# Classify only known transaction/service codes as non-merchandise.
# Alphanumeric StockCodes are often valid retail products and must not be flagged.
NON_MERCHANDISE_CODES = {
    "ADJUST", "ADJUST2", "AMAZONFEE", "B", "BANK CHARGES",
    "C2", "CRUK", "D", "DOT", "M", "POST", "S"
}
product_code_upper = products_df["product_id"].str.upper()
rule1_mask = (
    product_code_upper.isin(NON_MERCHANDISE_CODES)
    | product_code_upper.str.startswith("GIFT_0001_", na=False)
)

rule1_exceptions = products_df.loc[rule1_mask].copy()

rule1_df = create_exception_df(
    record_id=rule1_exceptions["product_id"],
    rule_failed="Rule 1: Non-Merchandise StockCode",
    observed_value=rule1_exceptions["product_name"],
    reason=(
        "Known fee, adjustment, discount, carriage, sample, or gift-voucher "
        "code; exclude from physical inventory and merchandise cost analysis."
    )
)

print("Rule 1 completed.")
print("Exceptions identified:", len(rule1_df))

display(rule1_df.head(10))


In [ ]:
orphan_payments = payments_df[
    ~payments_df["invoice_id"].isin(
        invoices_df["invoice_id"]
    )
].copy()

rule2_df = create_exception_df(
    record_id=orphan_payments["payment_id"],
    rule_failed="Rule 2: Orphan Payment",
    observed_value=orphan_payments["invoice_id"],
    reason=(
        "Payment reference does not exist in the "
        "invoices database table."
    )
)

print("Rule 2 completed.")
print("Orphan payments identified:", len(rule2_df))

display(rule2_df.head(10))

In [ ]:
# Aggregate payments by invoice
payment_summary = (
    payments_df
    .dropna(subset=["invoice_id", "payment_amount"])
    .groupby("invoice_id", as_index=False)
    .agg(
        total_paid=("payment_amount", "sum")
    )
)

# Merge payment totals with invoice amounts
invoice_reconciliation = invoices_df[
    ["invoice_id", "invoice_amount"]
].merge(
    payment_summary,
    on="invoice_id",
    how="left"
)

# Invoices without payments receive a zero payment total
invoice_reconciliation["total_paid"] = (
    invoice_reconciliation["total_paid"].fillna(0)
)

# Identify overpayments
overpayment_records = invoice_reconciliation[
    invoice_reconciliation["total_paid"]
    > invoice_reconciliation["invoice_amount"]
].copy()

# Calculate amount overpaid
overpayment_records["overpaid_amount"] = (
    overpayment_records["total_paid"]
    - overpayment_records["invoice_amount"]
)

print("Invoice reconciliation completed.")
print("Overpaid invoices:", len(overpayment_records))

display(overpayment_records.head(10))

In [ ]:
rule3_df = create_exception_df(
    record_id=overpayment_records["invoice_id"],
    rule_failed="Rule 3: Negative Outstanding Balance",
    observed_value=overpayment_records["overpaid_amount"].round(2),
    reason=(
        "Total payments credited exceed original invoice total, "
        "creating an unallocated credit balance."
    )
)

print("Rule 3 completed.")
print("Exceptions identified:", len(rule3_df))

display(rule3_df.head(10))

In [ ]:
# Non-merchandise rows have NULL unit_cost and are excluded from COGS validation.
product_cost_lookup = (
    products_df.loc[
        products_df["unit_cost"].notna(),
        ["product_id", "unit_cost"]
    ]
    .drop_duplicates(subset=["product_id"])
)

assert products_df["product_id"].is_unique, "Product IDs must be unique after trimming"
assert product_cost_lookup["product_id"].is_unique

print("Product cost lookup prepared.")
print("Products with synthetic costs:", len(product_cost_lookup))
print("Non-merchandise products excluded:", products_df["unit_cost"].isna().sum())


In [ ]:
orders_with_cost = orders_df.merge(
    product_cost_lookup,
    on="product_id",
    how="left",
    validate="many_to_one"
)

orders_with_cost["unit_gross_margin"] = (
    orders_with_cost["unit_selling_price"]
    - orders_with_cost["unit_cost"]
)

# A below-cost sale is a positive-quantity merchandise sale, not a return.
loss_records = orders_with_cost[
    orders_with_cost["quantity"].gt(0)
    & orders_with_cost["unit_cost"].notna()
    & orders_with_cost["unit_selling_price"].notna()
    & (
        orders_with_cost["unit_selling_price"].round(2)
        < orders_with_cost["unit_cost"].round(2)
    )
].copy()

print("Gross margin validation completed.")
print("Below-cost merchandise order lines:", len(loss_records))

display(loss_records.head(10))


In [ ]:
rule4_df = create_exception_df(
    record_id=loss_records["order_line_id"].astype("string"),

    rule_failed="Rule 4: Selling Below Unit Cost",

    observed_value=loss_records["unit_gross_margin"].round(2),

    reason=(
        "Positive-quantity merchandise sale is below its simulated unit cost; "
        "review as an illustrative margin exception."
    )
)

print("Rule 4 completed.")
print("Below-cost order lines:", len(rule4_df))

display(rule4_df.head(10))


In [ ]:
exceptions_df = pd.concat(
    [rule1_df, rule2_df, rule3_df, rule4_df],
    ignore_index=True
)

exceptions_df = exceptions_df[EXCEPTION_COLUMNS]

print("All financial exceptions consolidated successfully!")
print("Total exceptions:", len(exceptions_df))

display(exceptions_df.head(20))

In [ ]:
# Verify required columns
assert list(exceptions_df.columns) == EXCEPTION_COLUMNS

# Check resolution statuses
assert exceptions_df["resolution_status"].isin(
    ["Needs Review", "Resolved"]
).all()

# Check missing values
print("Missing values:")
print(exceptions_df.isnull().sum())

# Check duplicate exception records
duplicate_count = exceptions_df.duplicated(
    subset=["record_id", "rule_failed"]
).sum()

print("\nDuplicate exception records:", duplicate_count)

print("\nSchema validation passed!")

In [ ]:
exception_summary = (
    exceptions_df["rule_failed"]
    .value_counts()
    .reset_index()
)

exception_summary.columns = [
    "Validation Rule",
    "Exception Count"
]

display(exception_summary)

print("Total exceptions:", len(exceptions_df))

In [ ]:
exceptions_df = pd.concat(
    [
        rule1_df,
        rule2_df,
        rule3_df,
        rule4_df
    ],
    ignore_index=True
)

exceptions_df = exceptions_df[EXCEPTION_COLUMNS]

print("Total exceptions:", len(exceptions_df))

display(exceptions_df.head(10))

In [ ]:
# Check duplicate order IDs in original orders table

duplicate_order_ids = orders_df[
    orders_df.duplicated(
        subset=["order_id"],
        keep=False
    )
].sort_values("order_id")

print(
    "Total duplicate order rows:",
    len(duplicate_order_ids)
)

print(
    "Unique duplicated order IDs:",
    duplicate_order_ids["order_id"].nunique()
)

display(duplicate_order_ids.head(20))

In [ ]:
duplicate_order_summary = (
    duplicate_order_ids
    .groupby("order_id")
    .agg(
        row_count=("order_id", "size"),
        unique_products=("product_id", "nunique")
    )
    .reset_index()
)

display(duplicate_order_summary.head(20))

In [ ]:
record_id = loss_records["order_line_id"].astype("string")


In [ ]:
duplicate_count = exceptions_df.duplicated(
    subset=["record_id", "rule_failed"]
).sum()

print("Duplicate exception records:", duplicate_count)

assert duplicate_count == 0, "Duplicate exception records detected!"

print("Exception uniqueness validation passed!")

In [ ]:
exception_summary = (
    exceptions_df["rule_failed"]
    .value_counts()
    .reset_index()
)

exception_summary.columns = [
    "Validation Rule",
    "Exception Count"
]

display(exception_summary)

print("Total Exceptions:", len(exceptions_df))

In [ ]:
exception_file = DATA_DIR / "financial_exception_log.csv"

exceptions_df.to_csv(
    exception_file,
    index=False,
    encoding="utf-8"
)

print("Financial exception log exported successfully!")
print("Location:", exception_file)

In [ ]:
verified_exceptions_df = pd.read_csv(exception_file)

print("Exported rows:", len(verified_exceptions_df))
print("Exported columns:", verified_exceptions_df.columns.tolist())

display(verified_exceptions_df.head(10))

assert len(verified_exceptions_df) == len(exceptions_df)
assert verified_exceptions_df.columns.tolist() == EXCEPTION_COLUMNS

print("CSV verification successful!")

In [ ]:
documentation = f"""
# Financial Exception Log

## Overview

Automated validation using Python, Pandas, and PostgreSQL. New exceptions start as Needs Review.

## Exception Schema

| Column | Description |
|---|---|
| record_id | Unique affected record identifier; Rule 4 uses order_line_id |
| rule_failed | Validation rule that identified the record |
| observed_value | Value or source attribute that triggered the rule |
| reason | Explanation of the exception |
| resolution_status | Investigation status |

## Validation Results

| Rule | Exceptions |
|---|---:|
| Non-Merchandise StockCode | {len(rule1_df)} |
| Orphan Payment | {len(rule2_df)} |
| Negative Outstanding Balance | {len(rule3_df)} |
| Selling Below Unit Cost | {len(rule4_df)} |
| **Total** | **{len(exceptions_df)}** |

## Interpretation

- Alphanumeric StockCodes are not flagged simply because they contain letters; many are valid products.
- Known fee, adjustment, discount, carriage, sample, and gift-voucher codes are excluded from merchandise cost and inventory analysis.
- Below-cost checks apply only to positive-quantity merchandise lines and use simulated unit costs.
- Rule 4 is traceable using the unique order_line_id.
- These are review flags, not proof of actual financial loss. Costs are synthetic.

## Output

data/financial_exception_log.csv

## Technologies

- Python
- Pandas
- NumPy
- PostgreSQL
- SQLAlchemy
"""

documentation_file = DOCS_DIR / "exception_log.md"
documentation_file.write_text(documentation, encoding="utf-8")

print("Exception documentation updated!")
print("Location:", documentation_file)


05 - Financial Data Reconciliation

In [ ]:
constraint_audit = pd.read_sql_query("""
SELECT
    tc.table_name,
    tc.constraint_name,
    tc.constraint_type,
    STRING_AGG(
        kcu.column_name,
        ', ' ORDER BY kcu.ordinal_position
    ) AS columns

FROM information_schema.table_constraints tc

LEFT JOIN information_schema.key_column_usage kcu
    ON tc.constraint_catalog = kcu.constraint_catalog
    AND tc.constraint_schema = kcu.constraint_schema
    AND tc.constraint_name = kcu.constraint_name
    AND tc.table_name = kcu.table_name

WHERE tc.table_schema = 'public'
AND tc.table_name IN (
    'products',
    'orders',
    'invoices',
    'payments',
    'inventory_snapshots'
)

GROUP BY
    tc.table_name,
    tc.constraint_name,
    tc.constraint_type

ORDER BY
    tc.table_name,
    tc.constraint_type;
""", engine)

display(constraint_audit)

In [ ]:
order_quality = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_order_lines,
    COUNT(DISTINCT order_id) AS unique_orders,

    COUNT(*) FILTER (
        WHERE quantity > 0
    ) AS positive_quantity_lines,

    COUNT(*) FILTER (
        WHERE quantity < 0
    ) AS negative_quantity_lines,

    COUNT(*) FILTER (
        WHERE quantity = 0
    ) AS zero_quantity_lines,

    COUNT(*) FILTER (
        WHERE unit_selling_price < 0
    ) AS negative_price_lines,

    COUNT(*) FILTER (
        WHERE unit_selling_price = 0
    ) AS zero_price_lines,

    COUNT(*) FILTER (
        WHERE LEFT(order_id, 1) = 'C'
    ) AS cancellation_prefix_lines,

    SUM(quantity * unit_selling_price)
        AS total_net_sales,

    SUM(
        CASE
            WHEN quantity > 0
            THEN quantity * unit_selling_price
            ELSE 0
        END
    ) AS positive_line_sales,

    SUM(
        CASE
            WHEN quantity < 0
            THEN quantity * unit_selling_price
            ELSE 0
        END
    ) AS negative_quantity_value

FROM orders;
""", engine)

display(order_quality.T)

In [ ]:
product_quality = pd.read_sql_query("""

SELECT
    product_id,
    COUNT(*) AS record_count,

    COUNT(DISTINCT unit_cost)
        AS distinct_cost_count,

    COUNT(DISTINCT product_name)
        AS distinct_name_count,

    MIN(unit_cost) AS minimum_cost,
    MAX(unit_cost) AS maximum_cost

FROM products

GROUP BY product_id

HAVING COUNT(*) > 1;

""", engine)

display(product_quality)

In [ ]:
line_quality = pd.read_sql_query("""

WITH duplicate_combinations AS (

    SELECT
        order_id,
        product_id,
        COUNT(*) AS line_count

    FROM orders

    GROUP BY
        order_id,
        product_id

    HAVING COUNT(*) > 1
)

SELECT

    COUNT(*) AS repeated_order_product_groups,

    SUM(line_count - 1)
        AS additional_repeated_rows

FROM duplicate_combinations;

""", engine)

display(line_quality)

In [ ]:
cancellation_audit = pd.read_sql_query("""
SELECT
    CASE
        WHEN LEFT(order_id, 1) = 'C' AND quantity < 0
            THEN 'Cancellation prefix + negative quantity'
        WHEN LEFT(order_id, 1) = 'C' AND quantity > 0
            THEN 'Cancellation prefix + positive quantity'
        WHEN LEFT(order_id, 1) = 'C' AND quantity = 0
            THEN 'Cancellation prefix + zero quantity'
        WHEN LEFT(order_id, 1) <> 'C' AND quantity < 0
            THEN 'No cancellation prefix + negative quantity'
        ELSE 'Other'
    END AS transaction_category,

    COUNT(*) AS line_count,

    SUM(quantity * unit_selling_price) AS transaction_value

FROM orders
GROUP BY 1
ORDER BY line_count DESC;
""", engine)

display(cancellation_audit)

In [ ]:
duplicate_line_audit = pd.read_sql_query("""
SELECT
    order_id,
    product_id,
    COUNT(*) AS line_count,
    COUNT(DISTINCT quantity) AS distinct_quantities,
    COUNT(DISTINCT unit_selling_price) AS distinct_prices,
    MIN(quantity) AS minimum_quantity,
    MAX(quantity) AS maximum_quantity,
    MIN(unit_selling_price) AS minimum_price,
    MAX(unit_selling_price) AS maximum_price

FROM orders

GROUP BY order_id, product_id

HAVING COUNT(*) > 1

ORDER BY line_count DESC

LIMIT 20;
""", engine)

display(duplicate_line_audit)